## Sistem de detecție a intruziunilor bazat pe procesarea fluxurilor de date și Machine Learning

**Student:** Mihai Alexandru-Mario

**Grupa:** 405

**Disciplina:** Big Data

**An universitar:** 2025–2026

**Tehnologii utilizate:** Python, Apache Spark, Spark SQL, Spark MLlib, Apache Kafka

<a id="cuprins"></a>

### Cuprins

1. [Introducere](#introducere)

   * 1.1. [Prezentarea setului de date](#introducere-dataset)
   * 1.2. [Obiectivele proiectului](#introducere-obiective)

2. [Procesarea datelor](#procesare-date)

   * 2.1. [Încărcarea și validarea datelor](#incarcare-validare-date)
   * 2.2. [Preprocesarea datelor și câmpuri auxiliare](#preprocesare-date)
   * 2.3. [Pregătirea datelor pentru antrenare](#pregatire-date-antrenare)

     * 2.3.1. [Eliminarea încercărilor de atac](#eliminare-incercari-atac)
     * 2.3.2. [Eliminarea valorilor cu port/protocol 0](#eliminare-port-protocol-zero)
     * 2.3.3. [Simplificarea etichetelor și definirea categoriilor de atac](#simplificare-etichete-udf)
     * 2.3.4. [Eliminarea coloanelor irelevante](#eliminare-coloane-irelevante)
     * 2.3.5. [Conversia coloanelor numerice](#conversie-coloane-numerice)
     * 2.3.6. [Corectarea erorilor din dataset](#corectare-erori-dataset)
   * 2.4. [Salvarea setului de date procesat](#salvare-date-procesate)

3. [Metode ML și pipeline de antrenare](#ml-pipeline)

   * 3.1. [Încărcarea datelor procesate pentru ML](#incarcare-date-ml)
   * 3.2. [Împărțirea setului de date în train/test](#split-train-test)
   * 3.3. [Transformări Spark ML folosite în pipeline](#transformari-spark-ml)
   * 3.4. [Metrici de evaluare](#metrici-evaluare)
   * 3.5. [Logistic Regression](#logistic-regression)

     * 3.5.1. [Metrici Logistic Regression](#lr-metrici)
   * 3.6. [Random Forest](#random-forest)

     * 3.6.1. [Optimizarea hiperparametrilor](#rf-optimizare)
     * 3.6.2. [Configurația finală Random Forest](#rf-configuratie-finala)
     * 3.6.3. [Metrici Random Forest](#rf-metrici)
     * 3.6.4. [Importanța caracteristicilor](#rf-importanta-caracteristici)
   * 3.7. [Salvarea modelului și a fișierelor auxiliare](#salvare-model)

4. [Sistemul de detecție în timp real](#sistem-timp-real)

   * 4.1. [Descrierea sistemului](#descriere-sistem)
   * 4.2. [Integrarea Kafka, Spark Structured Streaming și modelul ML](#integrare-kafka-spark)
   * 4.3. [Diagramă sistem](#diagrama-sistem)
   * 4.4. [Structura folderelor sistemului](#structura-folderelor)

5. [Concluzii](#concluzii)

6. [Bibliografie](#bibliografie)


<a id="introducere"></a>

### 1. Introducere

<a id="introducere-dataset"></a>

#### 1.1 Prezentarea succintă a setului de date

În cadrul acestui proiect a fost utilizat setul de date **CICIDS2017**, publicat de Canadian Institute for Cybersecurity din cadrul University of New Brunswick [4]. Acesta este un dataset destinat cercetării și evaluării sistemelor de detecție a intruziunilor în rețea (IDS - intrusion detection system), fiind compus din trafic obișnuit (etichetat ca „BENIGN”) și scenarii de atac generate în scenarii controlate.

Dataset-ul a fost publicat de Sharafaldin, Lashkari și Ghorbani [3] cu scopul de a oferi un set de date mai realist pentru analiza traficului de rețea și pentru dezvoltarea sistemelor de tip Network Intrusion Detection System. Datele includ mai multe categorii de atacuri, precum atacuri de tip brute-force, denial-of-service (DoS), distributed denial-of-service (DDoS) și port scanning.

În proiect, datele au fost utilizate sub formă de fluxuri de rețea (generate din capturi de rețea PCAP și convertite cu software specializat - **CICFlowMeter** în CSV), fiecare înregistrare conținând caracteristici extrase din traficul capturat. Aceste caracteristici descriu proprietăți statistice ale comunicației dintre două calculatoare, precum durata fluxului, numărul de pachete, dimensiunea pachetelor, rata de transfer, flag-uri TCP și alte informații relevante pentru analiza comportamentului traficului.

Pentru îmbunătățirea calității datelor și pentru evitarea unor probleme cunoscute ale dataset-ului original, au fost studiate și lucrări care analizează erorile și inconsistențele din **CICIDS2017**, dar și software-ul **CICFlowMeter** [1], [2]. Aceste observații au fost importante în etapa de preprocesare și de streaming în timp real, fiind utilizate variantele îmbunătățite de cercetătorii de la **DistriNet Research Unit**, parte a **Department of Computer Science, KU Leuven** [6], [7].

**Link de descărcare** - [CICIDS2017_improved.zip](https://intrusion-detection.distrinet-research.be/CNS2022/Datasets/CICIDS2017_improved.zip)

<a id="introducere-obiective"></a>

#### 1.2 Enunțarea obiectivelor

Obiectivul principal al proiectului este dezvoltarea unui sistem de detecție a intruziunilor în traficul de rețea folosind tehnologii Big Data și algoritmi de Machine Learning. Sistemul urmărește clasificarea fluxurilor de rețea în trafic obișnuit sau în diferite categorii de atac, pe baza caracteristicilor extrase din datele **CICIDS2017**.

Obiectivele specifice ale proiectului sunt:

- încărcarea și analizarea setului de date;
- preprocesarea datelor prin eliminarea coloanelor irelevante, tratarea valorilor invalide și pregătirea caracteristicilor pentru antrenarea modelelor;
- definirea unor categorii de atac relevante pentru clasificare;
- antrenarea și evaluarea a doi algoritmi de Machine Learning - Logistic Regression și Random Forest;
- compararea performanței modelelor utilizate;
- construirea unui sistem de procesare care poate fi extins către detecția în timp real a traficului de rețea;
- integrarea modelului antrenat într-un flux de procesare bazat pe tehnologii precum Apache Spark Streaming și Apache Kafka.

Prin acest proiect se urmărește demonstrarea modului în care un pipeline Big Data poate fi utilizat pentru procesarea și clasificarea traficului de rețea, având ca scop identificarea automată a unor atacuri.

<a id="procesare-date"></a>

### 2. Procesarea datelor

<a id="incarcare-validare-date"></a>

#### 2.1 Încărcarea și validarea datelor

Setup mediu de lucru și import-uri

In [1]:
# doar pentru macOS - setare cale pentru Java 17 (necesar pentru Spark)
import os

os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [2]:
!java -version

openjdk version "17.0.19" 2026-04-21
OpenJDK Runtime Environment Homebrew (build 17.0.19+0)
OpenJDK 64-Bit Server VM Homebrew (build 17.0.19+0, mixed mode, sharing)


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import input_file_name, regexp_extract, col, when, isnan, count, udf
from pyspark.sql.types import StringType

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

import json

In [4]:
PARQUET_SAVE_DESTINATION = "processed_data/cicids2017_filtered.parquet"
DATASET_PATH = "raw_dataset/*.csv"

SAVE_DATA = True
DATA_SAVED_MODE = False

In [5]:
spark = SparkSession.builder.appName("ml-ids-bigdata") \
    .master("local[4]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "32") \
    .config("spark.default.parallelism", "4") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 16:08:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Citirea setului de date din fișierele CSV descărcate

In [6]:
df = spark.read.option("header", "true").option("inferSchema", "true").csv(DATASET_PATH)

Informații despre datele încărcate de Spark

In [7]:
print("rânduri totale:", df.count())
print("coloane totale:", len(df.columns))

rânduri totale: 2099976
coloane totale: 91


In [8]:
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- Flow ID: string (nullable = true)
 |-- Src IP: string (nullable = true)
 |-- Src Port: integer (nullable = true)
 |-- Dst IP: string (nullable = true)
 |-- Dst Port: integer (nullable = true)
 |-- Protocol: integer (nullable = true)
 |-- Timestamp: timestamp (nullable = true)
 |-- Flow Duration: integer (nullable = true)
 |-- Total Fwd Packet: integer (nullable = true)
 |-- Total Bwd packets: integer (nullable = true)
 |-- Total Length of Fwd Packet: integer (nullable = true)
 |-- Total Length of Bwd Packet: integer (nullable = true)
 |-- Fwd Packet Length Max: integer (nullable = true)
 |-- Fwd Packet Length Min: integer (nullable = true)
 |-- Fwd Packet Length Mean: double (nullable = true)
 |-- Fwd Packet Length Std: double (nullable = true)
 |-- Bwd Packet Length Max: integer (nullable = true)
 |-- Bwd Packet Length Min: integer (nullable = true)
 |-- Bwd Packet Length Mean: double (nullable = true)
 |-- Bwd Packet Length Std: double (nu

In [9]:
df.select("Flow ID", "Src IP", "Src Port", "Dst IP", "Dst Port", "Protocol").show(5, truncate=False)

+---------------------------------------+-------------+--------+-------------+--------+--------+
|Flow ID                                |Src IP       |Src Port|Dst IP       |Dst Port|Protocol|
+---------------------------------------+-------------+--------+-------------+--------+--------+
|192.168.10.50-192.168.10.3-56108-3268-6|192.168.10.50|56108   |192.168.10.3 |3268    |6       |
|192.168.10.50-192.168.10.3-42144-389-6 |192.168.10.50|42144   |192.168.10.3 |389     |6       |
|8.6.0.1-8.0.6.4-0-0-0                  |8.6.0.1      |0       |8.0.6.4      |0       |0       |
|192.168.10.25-224.0.0.251-5353-5353-17 |192.168.10.25|5353    |224.0.0.251  |5353    |17      |
|192.168.10.25-17.253.14.125-123-123-17 |192.168.10.25|123     |17.253.14.125|123     |17      |
+---------------------------------------+-------------+--------+-------------+--------+--------+
only showing top 5 rows


Grupare date după tipul de protocol

**Protocol 0** - eroare software CICFlowMeter, va fi eliminat

**Protocol 1** - ICMP

**Protocol 6** - TCP

**Protocol 17** - UDP

Creare vizualizare SQL pentru data frame-ul inițial

In [10]:
df.createOrReplaceTempView("initial_flows")

In [11]:
spark.sql("""
    SELECT Protocol, COUNT(*) AS count
    FROM initial_flows
    GROUP BY Protocol
    ORDER BY Protocol
""").show()

+--------+-------+
|Protocol|  count|
+--------+-------+
|       0|   1510|
|       1|    603|
|       6|1098771|
|      17| 999092|
+--------+-------+



Afișare tipuri de date în funcție de protocol

In [12]:
spark.sql("""
    SELECT
        CASE
            WHEN Protocol = 1 THEN 'ICMP'
            WHEN Protocol = 6 THEN 'TCP'
            WHEN Protocol = 17 THEN 'UDP'
            ELSE CAST(Protocol AS STRING)
        END AS protocol_name,
        Protocol AS protocol_id,
        Label,
        COUNT(*) AS flow_count
    FROM initial_flows
    WHERE Protocol IN (1, 6, 17)
    GROUP BY Protocol, Label
    ORDER BY Protocol, flow_count DESC
""").show(n=200, truncate=False)

+-------------+-----------+--------------------------------------+----------+
|protocol_name|protocol_id|Label                                 |flow_count|
+-------------+-----------+--------------------------------------+----------+
|ICMP         |1          |BENIGN                                |526       |
|ICMP         |1          |Infiltration - Portscan               |53        |
|ICMP         |1          |Portscan                              |24        |
|TCP          |6          |BENIGN                                |581677    |
|TCP          |6          |Portscan                              |159039    |
|TCP          |6          |DoS Hulk                              |158468    |
|TCP          |6          |DDoS                                  |95144     |
|TCP          |6          |Infiltration - Portscan               |71478     |
|TCP          |6          |DoS GoldenEye                         |7567      |
|TCP          |6          |Botnet - Attempted                   

In [13]:
spark.catalog.dropTempView("initial_flows")

True

<a id="preprocesare-date"></a>

#### 2.2 Preprocesarea datelor și câmpuri auxiliare

Adăugare fișier sursă pentru fiecare flow

In [14]:
df = df.withColumn("source_file", input_file_name())

Extrage ziua în care a fost generat fiecare flow din numele fișierului

In [15]:
df = df.withColumn("day",regexp_extract(col("source_file"), r"([^/\\]+)\.csv$", 1))

Validare cu fișierele CSV din dataset. Creare vizualizare nouă SQL pentru data frame, după adăugarea câmpurilor noi

In [16]:
df.createOrReplaceTempView("initial_flows_new")

In [17]:
spark.sql("""
    SELECT day, COUNT(*) AS count
    FROM initial_flows_new
    GROUP BY day
    ORDER BY day
""").show(truncate=False)

+---------+------+
|day      |count |
+---------+------+
|friday   |547557|
|monday   |371624|
|thursday |362076|
|tuesday  |322078|
|wednesday|496641|
+---------+------+



Vizualizare rânduri care au infinit pe coloana Flow Bytes/s (erori din aplicația CICFlowMeter)

In [18]:
spark.sql("""
    SELECT id, `Flow ID`, `Flow Bytes/s`
    FROM initial_flows_new
    WHERE `Flow Bytes/s` IN (
        CAST('Infinity' AS DOUBLE),
        CAST('-Infinity' AS DOUBLE)
    )
""").show(truncate=False)

+------+----------------------------------------+------------+
|id    |Flow ID                                 |Flow Bytes/s|
+------+----------------------------------------+------------+
|106984|192.168.10.12-192.168.10.19-137-137-17  |inf         |
|114891|192.168.10.12-192.168.10.17-137-137-17  |inf         |
|121883|192.168.10.12-192.168.10.17-137-137-17  |inf         |
|12013 |192.168.10.19-192.168.10.16-137-137-17  |inf         |
|465856|192.168.10.50-192.168.10.25-137-49234-17|inf         |
+------+----------------------------------------+------------+



In [19]:
spark.catalog.dropTempView("initial_flows_new")

True

<a id="pregatire-date-antrenare"></a>

#### 2.3 Pregătirea datelor pentru antrenare

<a id="eliminare-incercari-atac"></a>

##### 2.3.1 Eliminare încercări de atac

Setul de date îmbunătățit conține, pe lângă clasele de atac, și date marcate ca încercări de atac. Acestea sunt indicate prin coloana `Attempted Category`, care separă traficul asociat atacurilor reușite de traficul asociat unor încercări.

Pentru antrenarea modelului au fost păstrate doar fluxurile de trafic corespunzătoare atacurilor confirmate și traficului normal. Datele marcate ca încercări de atac au fost eliminate, deoarece pot introduce ambiguitate în procesul de învățare. Aceste fluxuri pot avea caracteristici apropiate atât de traficul normal, cât și de atacurile reale, ceea ce poate afecta calitatea etichetelor și performanța modelului.

In [20]:
df_no_attempted = df.filter(
    col("Attempted Category").cast("int") == -1
)

print("date inițiale:", df.count())
print("date eliminate:", df.count() - df_no_attempted.count())

date inițiale: 2099976


date eliminate: 11979


<a id="eliminare-port-protocol-zero"></a>

##### 2.3.2 Eliminare rânduri cu port 0 și protocol 0 (eroare în dataset)

În setul de date au fost identificate fluxuri cu valori `0` pentru porturi sau pentru protocol. Aceste valori au fost considerate irelevante pentru procesul de antrenare, deoarece nu descriu corect o conexiune de rețea utilizabilă pentru clasificarea traficului.

In [21]:
df_no_port0 = df_no_attempted.filter(
    (col("Src Port").cast("int") != 0) &
    (col("Dst Port").cast("int") != 0) &
    (col("Protocol").cast("int") != 0)
)

print("date inițiale:", df_no_attempted.count())
print("date eliminate:", df_no_attempted.count() - df_no_port0.count())

date inițiale: 2087997


date eliminate: 2113


<a id="simplificare-etichete-udf"></a>

##### 2.3.3 Simplificarea etichetelor și definirea categoriilor de atac (cu UDF)

Setul de date inițial conține mai multe etichete specifice pentru tipurile de atacuri. Pentru acest proiect, unele clase au fost grupate în categorii mai generale, astfel încât modelul să clasifice familii de atacuri, nu fiecare variantă. Această abordare simplifică problema de clasificare și face rezultatele mai ușor de interpretat.

Pentru realizarea acestei transformări a fost definită o funcție `map_attack_category`, aplicată asupra coloanei `Label` folosind UDF din Spark. Funcția verifică eticheta originală și o transformă într-o categorie generală de atac.

Clasele folosite în model au fost:

| Etichete originale | Categorie finală |
|---|---|
| BENIGN | BENIGN |
| FTP-Patator, SSH-Patator | BRUTE_FORCE |
| DoS Hulk, DoS GoldenEye | DOS_FLOODING |
| DoS Slowloris, DoS Slowhttptest | DOS_SLOW_HTTP |
| DDoS | DDOS |
| Portscan, Infiltration - Portscan | PORTSCAN |

Etichetele care nu au fost relevante pentru obiectivele proiectului au fost mapate în categoria `OTHER`, iar ulterior au fost eliminate din setul de date. Astfel, modelul a fost antrenat doar pe clasele considerate importante pentru clasificarea traficului de rețea în cadrul acestui proiect.

Această etapă ajută la reducerea complexității problemei de clasificare și la obținerea unor clase mai bine definite.


In [22]:
def map_attack_category(label):
    if label is None:
        return "OTHER"

    if label == "BENIGN":
        return "BENIGN"

    if label in [
        "FTP-Patator",
        "SSH-Patator"
    ]:
        return "BRUTE_FORCE"

    if label in [
        "DoS Hulk",
        "DoS GoldenEye"
    ]:
        return "DOS_FLOODING"

    if label in [
        "DoS Slowloris",
        "DoS Slowhttptest"
    ]:
        return "DOS_SLOW_HTTP"

    if label == "DDoS":
        return "DDOS"

    if label in [
        "Portscan",
        "Infiltration - Portscan"
    ]:
        return "PORTSCAN"

    return "OTHER"


map_attack_category_udf = udf(map_attack_category, StringType())

df_categorized_all = df_no_port0.withColumn(
    "attack_category",
    map_attack_category_udf(col("Label"))
)

df_categorized = df_categorized_all.filter(
    col("attack_category") != "OTHER"
)

initial_count = df_no_port0.count()
selected_count = df_categorized.count()

print("date inițiale:", initial_count)
print("date eliminate:", initial_count - selected_count)
print("date păstrate:", selected_count)

df_categorized.groupBy("Label", "attack_category") \
    .count() \
    .orderBy("attack_category", "Label") \
    .show(100, truncate=False)

date inițiale: 2085884
date eliminate: 887
date păstrate: 2084997


+-----------------------+---------------+-------+
|Label                  |attack_category|count  |
+-----------------------+---------------+-------+
|BENIGN                 |BENIGN         |1580532|
|FTP-Patator            |BRUTE_FORCE    |3972   |
|SSH-Patator            |BRUTE_FORCE    |2961   |
|DDoS                   |DDOS           |95144  |
|DoS GoldenEye          |DOS_FLOODING   |7567   |
|DoS Hulk               |DOS_FLOODING   |158468 |
|DoS Slowhttptest       |DOS_SLOW_HTTP  |1740   |
|DoS Slowloris          |DOS_SLOW_HTTP  |3859   |
|Infiltration - Portscan|PORTSCAN       |71714  |
|Portscan               |PORTSCAN       |159040 |
+-----------------------+---------------+-------+



<a id="eliminare-coloane-irelevante"></a>

##### 2.3.4 Eliminare coloane irelevante pentru antrenarea modelului

Înainte de antrenarea modelului, au fost eliminate coloanele care nu contribuie la clasificarea traficului sau care pot introduce bias în procesul de învățare. Scopul acestei etape este ca modelul să învețe pe baza caracteristicilor statistice ale fluxurilor, nu pe baza unor identificatori, adrese IP, porturi sau informații temporale specifice setului de date.

Coloanele eliminate au fost:

| Coloană | Motivul eliminării |
|---|---|
| id | Reprezintă un identificator numeric al rândului și nu conține informație relevantă pentru clasificare. |
| Flow ID | Este compus din adrese IP, porturi și protocol, deci poate introduce informații. |
| Src IP | A fost eliminat deoarece modelul ar putea învăța adresele IP sursă din dataset, ceea ce ar duce la bias și la o generalizare slabă pe trafic nou. |
| Dst IP | A fost eliminat din același motiv: modelul ar putea asocia anumite destinații cu anumite atacuri, în loc să învețe caracteristicile reale ale traficului. |
| Src Port | A fost eliminat pentru a evita dependența modelului de porturi specifice. |
| Dst Port | A fost eliminat deoarece anumite atacuri pot fi puternic asociate cu porturi specifice în dataset, ceea ce poate duce la memorarea scenariilor în locul învățării unor tipare generale. |
| Timestamp | A fost eliminat deoarece modelul ar putea învăța momentele exacte în care au fost simulate atacurile, în loc să folosească statistici relevante ale fluxurilor. |
| Label | A fost eliminat deoarece reprezintă eticheta originală a datelor. După maparea acesteia în coloana `attack_category`, coloana originală nu mai este necesară. |
| Attempted Category | A fost folosită doar pentru filtrarea încercărilor de atac și nu este necesară modelului. |
| source_file | A fost adăugată anterior pentru validarea citirii datelor și identificarea fișierului sursă, dar nu reprezintă o caracteristică a traficului. |
| day | A fost derivată din `source_file` și folosită doar pentru verificarea distribuției datelor pe zile, nu pentru antrenarea modelului. |

Prin eliminarea acestor coloane, modelul este forțat să utilizeze caracteristicile statistice generate de CICFlowMeter, precum dimensiunile pachetelor, durata fluxului, ratele de trafic sau flag-urile TCP. Această abordare reduce riscul de overfitting și ajută modelul să generalizeze mai bine pe trafic nou.

In [23]:
remove_columns = [
    "id",
    "Flow ID",
    "Src IP",
    "Dst IP",
    "Src Port",
    "Dst Port",
    "Timestamp",
    "Label",
    "Attempted Category",
    "source_file",
    "day",
]

df_final = df_categorized.drop(*remove_columns)

print("număr de rânduri final:", df_final.count())
print("număr de coloane final:", len(df_final.columns))

df_final.printSchema()

număr de rânduri final: 2084997
număr de coloane final: 83
root
 |-- Protocol: integer (nullable = true)
 |-- Flow Duration: integer (nullable = true)
 |-- Total Fwd Packet: integer (nullable = true)
 |-- Total Bwd packets: integer (nullable = true)
 |-- Total Length of Fwd Packet: integer (nullable = true)
 |-- Total Length of Bwd Packet: integer (nullable = true)
 |-- Fwd Packet Length Max: integer (nullable = true)
 |-- Fwd Packet Length Min: integer (nullable = true)
 |-- Fwd Packet Length Mean: double (nullable = true)
 |-- Fwd Packet Length Std: double (nullable = true)
 |-- Bwd Packet Length Max: integer (nullable = true)
 |-- Bwd Packet Length Min: integer (nullable = true)
 |-- Bwd Packet Length Mean: double (nullable = true)
 |-- Bwd Packet Length Std: double (nullable = true)
 |-- Flow Bytes/s: string (nullable = true)
 |-- Flow Packets/s: string (nullable = true)
 |-- Flow IAT Mean: double (nullable = true)
 |-- Flow IAT Std: double (nullable = true)
 |-- Flow IAT Max: inte

<a id="conversie-coloane-numerice"></a>

##### 2.3.5 Convertire coloane citite ca string în numeric

În urma citirii setului de date, unele coloane care conțin valori numerice au fost interpretate de Spark ca tip `string`.

Pentru antrenarea modelelor de machine learning, toate atributele folosite ca intrări trebuie să fie numerice. Din acest motiv, coloanele citite ca `string`, dar care reprezintă valori numerice, au fost convertite explicit la tipuri numerice.

In [24]:
string_numeric_columns = [
    "Flow Bytes/s",
    "Flow Packets/s"
]

for c in string_numeric_columns:
    if c in df_final.columns:
        df_final = df_final.withColumn(c, col(c).cast("double"))

<a id="corectare-erori-dataset"></a>

##### 2.3.6 Corectare erori set de date

În etapa de curățare a datelor au fost identificate valori infinite în coloanele `Flow Bytes/s` și `Flow Packets/s`. Acestea provin din erori de calcul ale aplicației CICFlowMeter.
Valorile `Infinity` și `-Infinity` nu pot fi folosite direct în procesul de antrenare, deoarece algoritmii de machine learning din Spark ML așteaptă valori numerice finite.

Pentru a corecta această problemă, rândurile care conțineau valori infinite în coloanele relevante au fost eliminate din setul de date. Această abordare a fost aleasă deoarece numărul acestor înregistrări era redus, iar păstrarea lor ar fi introdus valori artificiale în procesul de învățare.

In [25]:
invalid_rate_condition = (
    (col("Flow Bytes/s") == float("inf")) |
    (col("Flow Bytes/s") == float("-inf")) |
    (col("Flow Packets/s") == float("inf")) |
    (col("Flow Packets/s") == float("-inf"))
)

df_final.filter(invalid_rate_condition).select("Protocol", "Flow Duration", "Flow Bytes/s", "Flow Packets/s").show(50, truncate=False)

df_final = df_final.filter(~invalid_rate_condition)

+--------+-------------+------------+--------------+
|Protocol|Flow Duration|Flow Bytes/s|Flow Packets/s|
+--------+-------------+------------+--------------+
|17      |0            |Infinity    |Infinity      |
|17      |0            |Infinity    |Infinity      |
|17      |0            |Infinity    |Infinity      |
|17      |0            |Infinity    |Infinity      |
|17      |0            |Infinity    |Infinity      |
+--------+-------------+------------+--------------+



<a id="salvare-date-procesate"></a>

#### 2.4 Salvarea setului de date procesat

In [26]:
if SAVE_DATA:
    df_final.write.mode("overwrite").option("parquet.enable.dictionary", "false").option("compression", "snappy").parquet(PARQUET_SAVE_DESTINATION)

<a id="ml-pipeline"></a>

### 3. Metode ML și pipeline de antrenare

In [27]:
PARQUET_LOAD_PATH = "processed_data/cicids2017_filtered.parquet"
MODEL_SAVE_PATH = "models/rf_pipeline_model"
SHARED_FILES_PATH = "processed_data/json"

<a id="incarcare-date-ml"></a>

#### 3.1 Încărcarea datelor procesate pentru ML

Încărcare set de date din fișierele Parquet sau din dataframe-ul anterior (depinde de configurație)

In [28]:
if DATA_SAVED_MODE:
    df_model = spark.read.parquet(PARQUET_LOAD_PATH)

    print("randuri incarcate:", df_model.count())
    print("numar coloane:", len(df_model.columns))
else:
    df_model = df_final

In [29]:
if DATA_SAVED_MODE:
    df_model.printSchema()

In [30]:
target_col = "attack_category"

feature_columns = [
    field.name
    for field in df_model.schema.fields
    if field.name != target_col
]

print("coloane caracteristici:", len(feature_columns))
print(feature_columns)

coloane caracteristici: 82
['Protocol', 'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets', 'Total Length of Fwd Packet', 'Total Length of Bwd Packet', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd RST Flags', 'Bwd RST Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG F

<a id="split-train-test"></a>

#### 3.2 Împărțirea setului de date în train/test

Split train/test - comun ambelor modele pentru a putea evalua performanța pe același set

In [31]:
train_df, test_df = df_model.randomSplit([0.7, 0.3], seed=42)

print("nr. date antrenare:", train_df.count())
print("nr. date testare:", test_df.count())

train_df.groupBy(target_col).count().orderBy(target_col).show(truncate=False)
test_df.groupBy(target_col).count().orderBy(target_col).show(truncate=False)

nr. date antrenare: 1459517


nr. date testare: 625475


+---------------+-------+
|attack_category|count  |
+---------------+-------+
|BENIGN         |1106195|
|BRUTE_FORCE    |4879   |
|DDOS           |66605  |
|DOS_FLOODING   |116109 |
|DOS_SLOW_HTTP  |3944   |
|PORTSCAN       |161785 |
+---------------+-------+



+---------------+------+
|attack_category|count |
+---------------+------+
|BENIGN         |474332|
|BRUTE_FORCE    |2054  |
|DDOS           |28539 |
|DOS_FLOODING   |49926 |
|DOS_SLOW_HTTP  |1655  |
|PORTSCAN       |68969 |
+---------------+------+



<a id="transformari-spark-ml"></a>

#### 3.3 Transformări Spark ML folosite în pipeline

Pentru pregătirea datelor înainte de antrenarea modelelor au fost folosite mai multe transformări din Spark ML: `StringIndexer`, `VectorAssembler` la ambele, iar pentru Logistic Regression a fost utilizat și `StandardScaler`.

`StringIndexer` a fost folosit pentru transformarea etichetelor text din coloana `attack_category` în valori numerice. Algoritmii ML nu pot folosi direct etichete de tip string, astfel că fiecărei clase i-a fost atribuit un index numeric. Această etapă a fost comună ambelor modele, deoarece au nevoie de o coloană numerică pentru predicție.

`VectorAssembler` a fost folosit pentru combinarea tuturor coloanelor numerice de intrare într-o singură coloană de tip vector, numită `features`. Algoritmii ML primesc caracteristicile de intrare sub forma unui vector, această etapă fiind necesară pentru ambele modele.

`StandardScaler` a fost folosit doar pentru Logistic Regression. Acesta standardizează valorile numerice, aducând caracteristicile la o scară comparabilă. Această etapă este importantă pentru modelele liniare, deoarece acestea sunt sensibile la diferențele mari dintre atribute. De exemplu, coloane precum `Flow Duration` sau `Flow Bytes/s` pot avea valori mult mai mari decât coloanele de flag TCP. În schimb, modelele bazate pe arbori, precum Decision Tree și Random Forest, nu necesită standardizare, deoarece folosesc praguri de separare și sunt mai puțin influențate de scara numerică a caracteristicilor.

<a id="metrici-evaluare"></a>

#### 3.4 Metrici de evaluare

Pentru evaluarea modelelor au fost folosite două metrici principale: `accuracy` și `F1-score`.

**Accuracy** reprezintă proporția totală de predicții corecte din întregul set de test. Această metrică oferă o imagine generală asupra performanței modelului și este ușor de interpretat

**F1-score** combină precizia și recall-ul într-o singură metrică. Aceasta este mai potrivită pentru probleme de clasificare cu distribuție dezechilibrată a claselor, deoarece ia în considerare atât predicțiile fals pozitive, cât și predicțiile fals negative. În cadrul acestui proiect, F1-score este important deoarece unele tipuri de atacuri, precum `DOS_SLOW_HTTP` sau `BRUTE_FORCE`, au un număr mult mai mic de date decât clasa `BENIGN`.

Astfel, modelele nu au fost comparate doar pe baza acurateții globale, ci și pe baza scorului F1 și a matricei de confuzie. Acest lucru permite observarea comportamentului modelului pentru fiecare clasă de trafic, nu doar performanța pe întregul set de date.

<a id="logistic-regression"></a>

### 3.5 Logistic Regression

Acesta este un model liniar de clasificare, estimând probabilitatea ca un exemplu să aparțină unei anumite clase pe baza unei combinații liniare a caracteristicilor de intrare.

În acest proiect, modelul a fost folosit ca baseline pentru a compara performanța unui clasificator liniar cu un model bazat pe arbori, mai exact `Random Forest`.

Parametri model Logistic Regression

In [32]:
LR_MAX_ITER = 50
LR_REG_PARAM = 0.01
LR_ELASTIC_NET = 0.0
LR_FAMILY = "multinomial"

Definire pipeline model LR

In [33]:
label_indexer_lr = StringIndexer(
    inputCol=target_col,
    outputCol="label",
    handleInvalid="keep"
)

assembler_lr = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features",
    handleInvalid="keep"
)

scaler_lr = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withStd=True,
    withMean=False
)

lr = LogisticRegression(
    featuresCol="scaled_features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=LR_MAX_ITER,
    regParam=LR_REG_PARAM,
    elasticNetParam=LR_ELASTIC_NET,
    family=LR_FAMILY
)

pipeline_lr = Pipeline(stages=[
    label_indexer_lr,
    assembler_lr,
    scaler_lr,
    lr
])

<a id="lr-antrenare"></a>

#### Antrenare model

In [34]:
pipeline_model_lr = pipeline_lr.fit(train_df)

<a id="lr-testare"></a>

#### Testare model

In [35]:
predictions_lr = pipeline_model_lr.transform(test_df)

<a id="lr-etichete"></a>

#### Afișare etichete predicții

In [36]:
label_indexer_model_lr = pipeline_model_lr.stages[0]

print("Mapare etichete:")
for index, label in enumerate(label_indexer_model_lr.labels):
    print(f"{index} -> {label}")

Mapare etichete:
0 -> BENIGN
1 -> PORTSCAN
2 -> DOS_FLOODING
3 -> DDOS
4 -> BRUTE_FORCE
5 -> DOS_SLOW_HTTP


<a id="lr-metrici"></a>

#### 3.5.1 Metrici Logistic Regression

In [37]:
metrics_lr = ["accuracy", "f1"]

for metric in metrics_lr:
    evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName=metric
    )

    value = evaluator.evaluate(predictions_lr)
    print(f"{metric}: {value:.4f}")

accuracy: 0.9876


f1: 0.9860


<a id="lr-matrice-confuzie"></a>

#### Matrice de confuzie Logistic Regression

In [38]:
label_indexer_model_lr = pipeline_model_lr.stages[0]

label_mapping = {
    float(index): label
    for index, label in enumerate(label_indexer_model_lr.labels)
}

def prediction_to_category(prediction):
    if prediction is None:
        return "UNKNOWN"
    return label_mapping.get(float(prediction), "UNKNOWN")

prediction_to_category_udf = udf(prediction_to_category, StringType())

predictions_readable = predictions_lr.withColumn(
    "predicted_category",
    prediction_to_category_udf(col("prediction"))
)

predictions_readable.groupBy(target_col, "predicted_category") \
    .count() \
    .orderBy(target_col, "predicted_category") \
    .show(100, truncate=False)

+---------------+------------------+------+
|attack_category|predicted_category|count |
+---------------+------------------+------+
|BENIGN         |BENIGN            |470977|
|BENIGN         |BRUTE_FORCE       |19    |
|BENIGN         |DDOS              |66    |
|BENIGN         |DOS_FLOODING      |496   |
|BENIGN         |DOS_SLOW_HTTP     |37    |
|BENIGN         |PORTSCAN          |2737  |
|BRUTE_FORCE    |BENIGN            |893   |
|BRUTE_FORCE    |PORTSCAN          |1161  |
|DDOS           |BENIGN            |11    |
|DDOS           |DDOS              |28510 |
|DDOS           |DOS_FLOODING      |18    |
|DOS_FLOODING   |BENIGN            |761   |
|DOS_FLOODING   |DDOS              |368   |
|DOS_FLOODING   |DOS_FLOODING      |48772 |
|DOS_FLOODING   |DOS_SLOW_HTTP     |25    |
|DOS_SLOW_HTTP  |BENIGN            |440   |
|DOS_SLOW_HTTP  |DOS_SLOW_HTTP     |1128  |
|DOS_SLOW_HTTP  |PORTSCAN          |87    |
|PORTSCAN       |BENIGN            |571   |
|PORTSCAN       |BRUTE_FORCE    

<a id="lr-coeficienti"></a>

#### Analiză coeficienți pentru fiecare clasă

In [39]:
pipeline_model_lr_data = pipeline_model_lr.stages[-1]

print("număr clase:", pipeline_model_lr_data.numClasses-1)

coefficient_matrix = pipeline_model_lr_data.coefficientMatrix.toArray()

for class_index, class_name in enumerate(label_indexer_model_lr.labels):
    class_coefficients = coefficient_matrix[class_index]

    feature_coefficients = list(
        zip(feature_columns, class_coefficients)
    )

    top_positive = sorted(
        feature_coefficients,
        key=lambda x: x[1],
        reverse=True
    )[:10]

    top_negative = sorted(
        feature_coefficients,
        key=lambda x: x[1]
    )[:10]

    print("\n" + "=" * 80)
    print(f"clasa {class_index} -> {class_name}")

    print("\ntop coeficienți pozitivi:")
    for feature, coefficient in top_positive:
        print(f"{feature}: {coefficient:.6f}")

    print("\ntop coeficienți negativi:")
    for feature, coefficient in top_negative:
        print(f"{feature}: {coefficient:.6f}")

număr clase: 6

clasa 0 -> BENIGN

top coeficienți pozitivi:
Protocol: 0.469354
FIN Flag Count: 0.402121
Packet Length Min: 0.320070
Fwd Packet Length Max: 0.290550
Bwd Packet Length Min: 0.278036
Fwd Packet Length Std: 0.252886
Fwd Packet Length Mean: 0.214765
Fwd Segment Size Avg: 0.214765
Fwd Packet Length Min: 0.203831
Fwd Act Data Pkts: 0.183473

top coeficienți negativi:
Fwd Seg Size Min: -0.490112
Bwd RST Flags: -0.397588
Bwd Packet Length Std: -0.360234
Packet Length Variance: -0.329858
Packet Length Std: -0.263593
Bwd Bulk Rate Avg: -0.250543
Bwd Packet Length Max: -0.193648
Packet Length Max: -0.173405
SYN Flag Count: -0.171182
Subflow Fwd Packets: -0.154157

clasa 1 -> PORTSCAN

top coeficienți pozitivi:
Bwd RST Flags: 0.429291
Fwd Seg Size Min: 0.343324
Subflow Fwd Packets: 0.235141
Bwd Packets/s: 0.080956
Flow Packets/s: 0.070232
Down/Up Ratio: 0.060734
Fwd Packets/s: 0.054960
Active Std: 0.054798
Fwd URG Flags: 0.021168
URG Flag Count: 0.021126

top coeficienți negativi:


<a id="random-forest"></a>

### 3.6 Random Forest

Acesta este un model de clasificare bazat pe mai mulți arbori de decizie. Fiecare arbore face o predicție, iar rezultatul final este ales prin combinarea predicțiilor tuturor arborilor.

În acest proiect, algoritmul a fost folosit deoarece poate capta relații neliniare între caracteristicile traficului de rețea și este mai robust decât un singur `Decision Tree`. Prin combinarea mai multor arbori, modelul reduce riscul de overfitting și oferă predicții mai stabile.

In [40]:
label_indexer = StringIndexer(
    inputCol=target_col,
    outputCol="label",
    handleInvalid="keep"
)

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features",
    handleInvalid="keep"
)

<a id="rf-optimizare"></a>

#### 3.6.1 Optimizarea hiperparametrilor pentru Random Forest

Pentru optimizarea modelului Random Forest au fost testate mai multe valori pentru hiperparametrii `maxDepth` (8,10,12) și `numTrees` (50,100,150). Scopul a fost identificarea unor valori care oferă un compromis bun între performanță și complexitatea modelului.

In [41]:
import time

results = []

evaluator_accuracy = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

for max_depth in [8, 10, 12]:
    for no_trees in [50, 100, 150]:
        start_time = time.time()

        rf_tmp = RandomForestClassifier(
            featuresCol="features",
            labelCol="label",
            numTrees=no_trees,
            maxDepth=max_depth,
            seed=42
        )

        pipeline_tmp = Pipeline(stages=[
            label_indexer,
            assembler,
            rf_tmp
        ])

        model_tmp = pipeline_tmp.fit(train_df)
        predictions_tmp = model_tmp.transform(test_df)

        f1 = evaluator_f1.evaluate(predictions_tmp)
        accuracy = evaluator_accuracy.evaluate(predictions_tmp)

        duration = time.time() - start_time

        results.append((max_depth, no_trees, accuracy, f1, duration))

        print(
            f"maxDepth={max_depth}, "
            f"numTrees={no_trees}, "
            f"timp antrenare={duration:.2f}s"
        )

results_df = spark.createDataFrame(
    results,
    ["maxDepth", "numTrees", "accuracy", "f1", "duration_seconds"]
)

results_df.orderBy("f1", ascending=False).show(truncate=False)

maxDepth=8, numTrees=50, timp antrenare=50.43s


maxDepth=8, numTrees=100, timp antrenare=66.82s


maxDepth=8, numTrees=150, timp antrenare=87.26s


maxDepth=10, numTrees=50, timp antrenare=52.64s


maxDepth=10, numTrees=100, timp antrenare=79.93s


maxDepth=10, numTrees=150, timp antrenare=105.80s


maxDepth=12, numTrees=50, timp antrenare=57.87s


maxDepth=12, numTrees=100, timp antrenare=93.45s


maxDepth=12, numTrees=150, timp antrenare=126.90s
+--------+--------+------------------+------------------+------------------+
|maxDepth|numTrees|accuracy          |f1                |duration_seconds  |
+--------+--------+------------------+------------------+------------------+
|12      |100     |0.9997665774011751|0.9997664909277078|93.45399403572083 |
|12      |150     |0.9997569846916343|0.9997569083630626|126.89659190177917|
|12      |50      |0.9997441944122467|0.9997441127199362|57.86852192878723 |
|10      |150     |0.9993061273432191|0.9993051246624856|105.79598999023438|
|10      |100     |0.999294935848755 |0.9992938730276419|79.92628693580627 |
|10      |50      |0.999294935848755 |0.9992934554205544|52.64232087135315 |
|8       |50      |0.9990806986690115|0.9990786081047228|50.425589084625244|
|8       |150     |0.9990359326911548|0.9990334801682228|87.26299595832825 |
|8       |100     |0.9989703825092929|0.9989675274154829|66.82324004173279 |
+--------+--------+-------

<a id="rf-configuratie-finala"></a>

#### 3.6.2 Alegerea configurației finale pentru Random Forest

Cel mai mare scor a fost obținut de modelul cu `maxDepth=12` și `numTrees=100`, în urma testării parametrilor, dar din cauza timpului mai mare de antrenare, a fost ales `numTrees=50` pentru că oferă un rezultat aproape identic, cu un timp mai scurt de antrenare.

In [42]:
RF_TREES = 50
RF_MAX_DEPTH = 12
RF_SEED = 42

<a id="rf-pipeline"></a>

#### Pipeline model Random Forest

In [43]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    numTrees=RF_TREES,
    maxDepth=RF_MAX_DEPTH,
    seed=RF_SEED
)

pipeline_rf = Pipeline(stages=[
    label_indexer,
    assembler,
    rf
])

<a id="rf-antrenare"></a>

#### Antrenare model

In [44]:
rf_model = pipeline_rf.fit(train_df)

<a id="rf-testare"></a>

#### Testare model

In [45]:
predictions = rf_model.transform(test_df)

predictions.select(
    target_col,
    "label",
    "prediction",
    "probability"
).show(5, truncate=False)

+---------------+-----+----------+------------------------------------------------------------+
|attack_category|label|prediction|probability                                                 |
+---------------+-----+----------+------------------------------------------------------------+
|BENIGN         |0.0  |0.0       |[0.9999088458827721,9.11541172279462E-5,0.0,0.0,0.0,0.0,0.0]|
|BENIGN         |0.0  |0.0       |[0.9999088458827721,9.11541172279462E-5,0.0,0.0,0.0,0.0,0.0]|
|BENIGN         |0.0  |0.0       |[0.9999088458827721,9.11541172279462E-5,0.0,0.0,0.0,0.0,0.0]|
|BENIGN         |0.0  |0.0       |[0.9999088458827721,9.11541172279462E-5,0.0,0.0,0.0,0.0,0.0]|
|BENIGN         |0.0  |0.0       |[0.9999088458827721,9.11541172279462E-5,0.0,0.0,0.0,0.0,0.0]|
+---------------+-----+----------+------------------------------------------------------------+
only showing top 5 rows


<a id="rf-etichete"></a>

#### Afișare etichete predicții

In [46]:
label_indexer_model_rf = rf_model.stages[0]

print("Mapare etichete:")
for index, label in enumerate(label_indexer_model_rf.labels):
    print(f"{index} -> {label}")

Mapare etichete:
0 -> BENIGN
1 -> PORTSCAN
2 -> DOS_FLOODING
3 -> DDOS
4 -> BRUTE_FORCE
5 -> DOS_SLOW_HTTP


<a id="rf-metrici"></a>

#### 3.6.3 Metrici Random Forest

In [47]:
metrics = ["accuracy", "f1"]

for metric in metrics:
    evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName=metric
    )

    value = evaluator.evaluate(predictions)
    print(f"{metric}: {value:.4f}")

accuracy: 0.9997


f1: 0.9997


<a id="rf-matrice-confuzie"></a>

#### Matrice de confuzie Random Forest

In [48]:
label_mapping = {
    float(index): label
    for index, label in enumerate(label_indexer_model_rf.labels)
}

def prediction_to_category(prediction):
    if prediction is None:
        return "UNKNOWN"
    return label_mapping.get(float(prediction), "UNKNOWN")

prediction_to_category_map_udf = udf(prediction_to_category, StringType())

predictions_labeled = predictions.withColumn(
    "predicted_category",
    prediction_to_category_map_udf(col("prediction"))
)

predictions_labeled.groupBy(target_col, "predicted_category") \
    .count() \
    .orderBy(target_col, "predicted_category") \
    .show(100, truncate=False)

+---------------+------------------+------+
|attack_category|predicted_category|count |
+---------------+------------------+------+
|BENIGN         |BENIGN            |474326|
|BENIGN         |DOS_FLOODING      |1     |
|BENIGN         |PORTSCAN          |5     |
|BRUTE_FORCE    |BENIGN            |10    |
|BRUTE_FORCE    |BRUTE_FORCE       |2044  |
|DDOS           |DDOS              |28539 |
|DOS_FLOODING   |BENIGN            |65    |
|DOS_FLOODING   |DOS_FLOODING      |49854 |
|DOS_FLOODING   |DOS_SLOW_HTTP     |7     |
|DOS_SLOW_HTTP  |BENIGN            |11    |
|DOS_SLOW_HTTP  |DOS_SLOW_HTTP     |1644  |
|PORTSCAN       |BENIGN            |61    |
|PORTSCAN       |PORTSCAN          |68908 |
+---------------+------------------+------+



<a id="rf-importanta-caracteristici"></a>

#### 3.6.4 Verificare importanță caracteristici

In [49]:
rf_model_data = rf_model.stages[-1]

feature_importance = list(
    zip(feature_columns, rf_model_data.featureImportances.toArray())
)

feature_importance_s = sorted(
    feature_importance,
    key=lambda x: x[1],
    reverse=True
)

print("Top 15 caracteristici:")
for feature, importance in feature_importance_s[:15]:
    print(f"{feature}: {importance:.6f}")

Top 15 caracteristici:
Total Length of Fwd Packet: 0.072566
Fwd Packet Length Max: 0.070880
Packet Length Std: 0.069109
Packet Length Variance: 0.061885
Bwd Packet Length Std: 0.057270
Flow IAT Max: 0.051271
Bwd Packet Length Max: 0.050626
Flow Duration: 0.050173
Flow Bytes/s: 0.044899
Fwd Packets/s: 0.037332
RST Flag Count: 0.033313
Average Packet Size: 0.028013
Packet Length Max: 0.027854
Subflow Bwd Bytes: 0.023711
Packet Length Mean: 0.023278


<a id="salvare-model"></a>

#### 3.7 Salvarea modelului și a fișierelor auxiliare

Salvare model, coloane caracteristici pentru model și etichete predicții ca JSON

In [50]:
if SAVE_DATA:
    rf_model.write().overwrite().save(MODEL_SAVE_PATH)
    
    os.makedirs(SHARED_FILES_PATH, exist_ok=True)

    # salvare caracteristici folosite de VectorAssembler ca json
    with open(f"{SHARED_FILES_PATH}/feature_columns.json", "w") as f:
        json.dump(feature_columns, f, indent=2)

    # salvare mapare etichete pentru model ca json
    label_indexer_model_rf = rf_model.stages[0]
    label_names_rf = list(label_indexer_model_rf.labels)

    with open(f"{SHARED_FILES_PATH}/label_names.json", "w") as f:
        json.dump(label_names_rf, f, indent=2)

    print("Fișierele necesare pentru model au fost salvate.")

Fișierele necesare pentru model au fost salvate.


<a id="sistem-timp-real"></a>

### 4. Sistemul de detecție în timp real folosind Kafka și Spark Structured Streaming

<a id="descriere-sistem"></a>

#### 4.1 Descrierea sistemului

Sistemul propus implementează o arhitectură containerizată pentru monitorizarea, procesarea și clasificarea traficului de rețea, fiind organizat în două stack-uri Docker Compose separate. Primul stack reprezintă rețeaua monitorizată, în cadrul căreia este simulat trafic normal, dar și scenarii de atac. Al doilea stack este responsabil de procesarea datelor și de aplicarea modelului ML asupra traficului.

Primul stack Docker Compose definește infrastructura de rețea monitorizată prin intermediul unei rețele interne Docker, accesibilă containerelor. Rețeaua include un container server, ce rulează o aplicație Node.js, o bază de date Postgres 16, un container client care generează trafic normal către server utilizând un script shell și `cURL`, precum și un container dedicat simulării atacurilor asupra serverului. Traficul generat în această rețea este capturat cu ajutorul aplicației `tcpdump`, care rulează în ferestre de 60 de secunde. Pentru fiecare interval este generat un fișier PCAP, salvat în foldetr-ul `data/input_pcap`, urmând ca acesta să fie preluat ulterior de componenta de procesare.

Al doilea stack Docker Compose conține componentele necesare pentru procesarea, transmiterea și clasificarea traficului capturat. În prima etapă, un script monitorizează folder-ul în care sunt salvate fișierele PCAP și adaugă fiecare fișier nou într-o coadă de procesare. Fișierele sunt procesate secvențial, iar pentru fiecare dintre acestea este lansat CICFlowMeter, instrument utilizat pentru extragerea caracteristicilor statistice la nivel de flux de rețea. Rezultatul acestei procesări constă în fișiere CSV care conțin flow-uri de rețea într-un format similar cu cel utilizat în etapa de antrenare a modelului. Aceste fișiere sunt salvate în folder-ul `data/input_csv`.

<a id="integrare-kafka-spark"></a>

#### 4.2 Integrarea Kafka, Spark Structured Streaming și modelul ML

Ulterior, componenta Kafka monitorizează directorul de ieșire al CICFlowMeter și utilizează un mecanism similar de coadă pentru preluarea fișierelor CSV generate. Datele sunt transmise printr-un topic Kafka, de unde sunt consumate de aplicația Spark Structured Streaming. În cadrul acestei componente, datele primite sunt preprocesate conform aceleiași structuri utilizate în etapa de antrenare, iar modelul ML salvat anterior este aplicat asupra fluxurilor de rețea în timp real.

Rezultatele clasificării sunt afișate în consolă sumarizat, afișând clasele prezise din fiecare fișier, iar în cazul unui atac se afișează IP sursă și IP destinație. În același timp, predicțiile sunt salvate în directorul `spark_output/` sub formă de fișiere Parquet, pentru a permite analiza ulterioară a detecțiilor.

Separarea arhitecturii în două stack-uri Docker Compose permite delimitarea dintre mediul monitorizat și pipeline-ul de procesare al datelor. Această abordare facilitează izolarea componentelor, testarea independentă a fluxului de captură și procesare, precum și înlocuirea sau extinderea ulterioară a unor module, precum generatorul de trafic, componenta de extragere a caracteristicilor sau modelul de clasificare.

> **Notă privind mediul de testare și utilizarea responsabilă**
>
> Toate operațiunile de captură, analiză și simulare de trafic prezentate în acest proiect sunt efectuate exclusiv într-un mediu izolat și strict controlat, folosind o rețea virtuală Docker.
>
> Pentru validarea funcționării sistemului de clasificare au fost simulate scenarii controlate, rulate doar în rețeaua Docker. Au fost simulate atacuri precum port-scan și DoS asupra containerului server. Script-urile de simulare ale atacurilor și de capturarea traficului nu au fost publicate în repository, deoarece folosesc instrumente precum `tcpdump` și `nmap`, dar și altele, care trebuie utilizate numai în medii izolate și strict controlate.
>
> Proiectul are caracter educațional și experimental, iar toate testele au fost realizate exclusiv pe infrastructură proprie, fără a viza sisteme externe.

<a id="diagrama-sistem"></a>

#### 4.3 Diagramă sistem

<style>
.diagram-print-block {
  display: block;
  clear: both;
  width: 100%;
  margin: 24px auto 48px auto;
  text-align: center;
}

.diagram-print-image {
  display: block;
  max-width: 100%;
  max-height: 75vh;
  height: auto;
  margin: 0 auto;
}

@media print {
  .diagram-print-block {
    break-inside: avoid;
    page-break-inside: avoid;
    break-after: page;
    page-break-after: always;
  }

  .diagram-print-image {
    max-width: 180mm;
    max-height: 230mm;
    object-fit: contain;
  }
}
</style>

<figure class="diagram-print-block">
  <img src="./diagram.png" alt="Arhitectura pipeline-ului IDS" class="diagram-print-image">
</figure>

<a id="structura-folderelor"></a>

#### 4.4 Structura folderelor sistemului

```text
ProiectBigData2026/
├── data/
│   ├── captures/
│   │   ├── failed_pcap/          # fișiere PCAP care nu au fost procesate corect
│   │   ├── input_csv/            # fișiere CSV generate din PCAP care sunt procesate ulterior de Kafka
│   │   ├── input_pcap/           # fișiere PCAP salvate din traficul de rețea
│   │   ├── processed_pcap/       # fișiere PCAP procesate de CICFlowMeter (pentru arhivare)
│   │   └── tmp/                  # folder fișiere temporare
│   │
│   └── processed_csv/            # fișiere CSV procesate de sistem (pentru arhivare)
├── docker-network/
│   ├── api-service/              # serviciu API pentru testare
│   ├── attacker-simulation/      # script-uri pentru simulare atacuri controlate
│   ├── client-traffic-generator/ # script pentru generare trafic normal
│   ├── sniffer-container/        # script pentru captare trafic din rețea
│   └── docker-compose.yml        # fișier Compose pentru pornire containere
│
├── live-ids/
│   ├── kafka-producer/           # script procesare CSV-uri și trimitere către Spark în timp real
│   ├── pcap-processor/           # script procesare fișiere PCAP (pornește CICFlowMeter)
│   ├── spark-ml/                 # script Spark Streaming și aplicare model în timp real
│   └── docker-compose.yml        # fișier Compose pentru pornire containere
│
├── raw_dataset/                  # set de date inițial (CICIDS2017-improved), fișiere CSV    
│
├── models/                       # fișiere model ML antrenat
│
├── processed_data/               # set de date procesat (output Notebook), json cu date pentru script Spark Streaming
│
├── spark_output/                 # log-uri din script-ul Spark Streaming
│
├── proiect.ipynb                 # fișier Notebook pentru procesare date și antrenare model
├── README.md                     # documentație proiect pe Git
└── .gitignore                    # fișier gitignore -> nu trebuie încărcate modelul și datele procesate pe Git
```

<a id="concluzii"></a>

### 5. Concluzii

În cadrul proiectului a fost implementat un sistem IDS experimental, bazat pe o arhitectură containerizată și pe un pipeline de procesare al datelor în timp real. Sistemul integrează capturarea traficului de rețea cu `tcpdump`, extragerea caracteristicilor folosind `CICFlowMeter`, trimiterea datelor prin Apache Kafka și clasificarea acestora cu Apache Spark Structured Streaming și MLlib, utilizând un model antrenat anterior.

Rezultatele obținute în etapa de testare arată că modelul identifică tiparele existente în setul de date utilizat. Totuși, deoarece datele de test provin din aceeași distribuție ca datele de antrenare, performanța obținută reflectă în principal comportamentul modelului în contextul dataset-ului, nu neapărat într-un mediu real. În testarea live, traficul este generat într-o rețea Docker, folosind aplicații și instrumente diferite față de cele utilizate pentru CICIDS2017, ceea ce poate conduce la diferențe în caracteristicile flow-urilor și în rezultatele clasificării. Această diferență este accentuată de faptul că CICIDS2017 a fost generat într-un mediu controlat, cu trafic și atacuri produse sintetic.

O altă limitare a sistemului este dependența de instrumente externe care nu oferă o integrare directă prin API-uri Python. În cazul CICFlowMeter, procesarea fișierelor PCAP a fost realizată printr-un wrapper care apelează aplicația din linia de comandă. Acesta monitorizează apariția fișierelor noi, le pregătește pentru procesare într-un folder temporar și mută fișierele generate ulterior.

Deși această abordare permite integrarea CICFlowMeter în pipeline, ea introduce complexitate suplimentară și un consum mai mare de resurse, deoarece implică operații repetate asupra fișierelor. Din acest motiv, a fost necesară implementarea unor mecanisme suplimentare pentru monitorizarea folderelor, gestionarea cozilor de procesare și verificarea faptului că fișierele au fost scrise complet înainte de a fi procesate.

În ciuda limitărilor, proiectul demonstrează implementarea unui flux complet pentru analiza traficului de rețea, acoperind etapele de captură, extragere a caracteristicilor, transmitere, clasificare și salvarea rezultatelor. Arhitectura evidențiază modul în care tehnologii precum Docker, Kafka, Spark Structured Streaming și algoritmii de învățare automată pot fi integrate într-un sistem funcțional de tip IDS (intrusion detection system).

Prin separarea componentelor în containere și prin utilizarea unui pipeline de procesare în timp real, sistemul oferă o bază pentru experimente ulterioare. Pe viitor, proiectul poate fi extins prin generarea unui trafic mai variat, definirea unei rețele mai complexe și pregătirea unui set de date propriu, mai apropiat de condițiile reale, care să permită o evaluare mai bună a comportamentului sistemului într-un mediu real.

<a id="bibliografie"></a>

### 6. Bibliografie

- [1] L. Liu, G. Engelen, T. Lynar, D. Essam și W. Joosen, „Error Prevalence in NIDS Datasets: A Case Study on CIC-IDS-2017 and CSE-CIC-IDS-2018”, în 2022 IEEE Conference on Communications and Network Security (CNS), 2022, pp. 254–262. Link: (https://intrusion-detection.distrinet-research.be/CNS2022/index.html)

- [2] G. Engelen, V. Rimmer și W. Joosen, „Troubleshooting an Intrusion Detection Dataset: The CICIDS2017 Case Study”, în 2021 IEEE Security and Privacy Workshops (SPW), 2021, pp. 7–12. Link: (https://intrusion-detection.distrinet-research.be/WTMC2021/tools_datasets.html)

- [3] I. Sharafaldin, A. H. Lashkari și A. A. Ghorbani, „Toward Generating a New Intrusion Detection Dataset and Intrusion Traffic Characterization”, în *International Conference on Information Systems Security and Privacy*, 2018. Link: (https://api.semanticscholar.org/CorpusID:4707749)

- [4] Canadian Institute for Cybersecurity, University of New Brunswick, „CICIDS2017 Dataset”, University of New Brunswick. Link: https://www.unb.ca/cic/datasets/ids-2017.html

- [5] R. Panigrahi și S. Borah, „A Detailed Analysis of CICIDS2017 Dataset for Designing Intrusion Detection Systems”, International Journal of Engineering & Technology, vol. 7, pp. 479–482, 2018. Link: (https://www.researchgate.net/publication/329045441_A_detailed_analysis_of_CICIDS2017_dataset_for_designing_Intrusion_Detection_Systems)

- [6] G. Engelen, „CICFlowMeter — fixed version of the original CICFlowMeter tool”, GitHub repository. Link: (https://github.com/GintsEngelen/CICFlowMeter)

- [7] G. Engelen, „WTMC2021-Code”, GitHub repository. Link: (https://github.com/GintsEngelen/WTMC2021-Code)